# Use case 3: notify DestinE users when Extremes-DT data lands on the data bridge

**Ask (Emma Kuwertz, Matt Griffith, possibly ECPDS):** the Extremes Digital Twin (DTE) runs on
ATOS at ECMWF. Its output is transferred to the DestinE data bridges (LUMI today) and kept there
until it is **15 days old**. DestinE users, businesses in particular, want to be told the moment a
field has been transferred, so they can fetch it while it is still there.

This notebook shows the consumer side of that flow with the **Aviso 2 client** (`pyaviso` 2.x):
what the `data` notification on the LUMI Aviso 2 server tells a user, how to derive the
**availability window** from it, and how to wire it into business systems (webhook, Teams,
command, resume state, CLI listener, a local availability tracker).

```
ATOS (DTE run)  ──transfer──►  LUMI data bridge (FDB / Polytope)
                                     │  publish "data" event
                                     ▼
                       Aviso 2  https://aviso2.lumi.apps.dte.destination-earth.eu
                                     │  watch / replay (Bearer = DESP token)
                                     ▼
                     business workflow: fetch via Polytope within 15 days
```

| Setting | Value |
| --- | --- |
| Aviso server | `https://aviso2.lumi.apps.dte.destination-earth.eu` (override with `AVISO_BASE_URL`) |
| Event type | `data`, MARS-like identifiers (`class`, `stream`, `type`, `levtype`, `date`, `time`, `step`, ...) |
| Aviso auth | Bearer token: the **DESP offline token** in `~/.polytopeapirc` (`user_key`, written by `desp-authentication.py`) or `AVISO_TOKEN` |
| Data | Polytope `polytope.lumi.apps.dte.destination-earth.eu`, dataset `extremes-dt`, ~15-day rolling window |
| Client | `pip install pyaviso earthkit-data pandas pyyaml` |

> **Hostname.** The server answers at `aviso2.lumi.apps.dte.destination-earth.eu`. The same name
> without `.dte.` does not resolve, and `aviso.lumi.apps.dte.destination-earth.eu` (no `2`) is
> the old Aviso 1.x server (`pyaviso==1.0.2`, see `../../DE/aviso-examples/`).

Sections:

1. Connect with the DESP token
2. What a `data` notification tells a business user
3. What is on the data bridge right now: tail replay, transfer lag, expiry
4. Check against Polytope: inside and outside the 15-day window
5. Integration patterns: webhook, Teams, command, resume, CLI listener
6. A minimal availability tracker
7. Live listen (time-bounded)
8. Publisher side: who raises the event

## 1. Connect with the DESP token

The LUMI Aviso 2 server is fronted by DESP's identity provider, so the **same offline token that
Polytope uses** (`~/.polytopeapirc`) is accepted as a Bearer token. `AVISO_TOKEN` wins if set.
The schema listing is public; watch and replay need the token. Nothing secret is printed.

In [1]:
# %pip install -q pyaviso earthkit-data pandas pyyaml

import json
import os
import pathlib
import queue
import threading
import time
from datetime import datetime, timedelta, timezone

import pandas as pd
import pyaviso

BASE_URL = os.environ.get("AVISO_BASE_URL", "https://aviso2.lumi.apps.dte.destination-earth.eu").rstrip("/")
EVENT = "data"
RETENTION_DAYS = 15                       # data bridge retention for Extremes-DT output
POLYTOPE_ADDRESS = "polytope.lumi.apps.dte.destination-earth.eu"
DATASET = "extremes-dt"
OUT_DIR = pathlib.Path("output") / "uc03"
OUT_DIR.mkdir(parents=True, exist_ok=True)
POLYTOPE_RC = pathlib.Path.home() / ".polytopeapirc"


def resolve_token():
    if os.environ.get("AVISO_TOKEN"):
        return os.environ["AVISO_TOKEN"], "AVISO_TOKEN"
    if POLYTOPE_RC.exists():
        key = json.loads(POLYTOPE_RC.read_text(encoding="utf-8")).get("user_key")
        if key:
            return key, "DESP offline token (~/.polytopeapirc)"
    raise RuntimeError("No token: set AVISO_TOKEN or run ../../DE/aviso-examples/desp-authentication.py")


TOKEN, token_source = resolve_token()
client = pyaviso.AvisoClient(base_url=BASE_URL, auth=pyaviso.Bearer(TOKEN))
print("pyaviso", pyaviso.__version__, "| server", client.base_url, "| auth:", token_source)


def open_stream(client, event_type, *, filter, start=None, mode=None, triggers=None):
    # pyaviso 2.1.0 names the cursor `start_from=`; newer builds `from_=`.
    kwargs = {"filter": filter}
    if mode is not None:
        kwargs["mode"] = mode
    if triggers is not None:
        kwargs["triggers"] = triggers
    if start is None:
        return client.listen(event_type, **kwargs)
    try:
        return client.listen(event_type, start_from=start, **kwargs)
    except TypeError:
        return client.listen(event_type, from_=start, **kwargs)

pyaviso 2.1.0 | server https://aviso2.lumi.apps.dte.destination-earth.eu/ | auth: DESP offline token (~/.polytopeapirc)


## 2. What a `data` notification tells a business user

The schema is owned by the server. Only `class` is required in a filter; every other identifier
narrows the match. On this stream the **payload is optional and in practice absent**: the
identifiers *are* the message. Read one as "the field `class=d1 stream=... levtype=... date=...
time=... step=...` is now on the data bridge", and combine it with the retention rule to get
"...and will disappear on `date + 15 days`".

Identifier values arrive as **strings** (`"step": "10"`, `"time": "0000"`), which is what
Polytope/MARS requests expect anyway.

In [2]:
schema = client.schema_for(EVENT).schema
rows = [{"identifier": k, "type": v.get("type"), "required": v.get("required", False),
         "values / format": v.get("values") or v.get("canonical_format") or v.get("range") or v.get("default") or ""}
        for k, v in schema["identifier"].items()]
display(pd.DataFrame(rows).set_index("identifier"))
print("payload required:", schema.get("payload", {}).get("required"))

,type,required,values / format
identifier,,,
class,StringHandler,True,
date,DateHandler,False,%Y%m%d
domain,EnumHandler,False,[g]
expver,ExpverHandler,False,0001
levtype,EnumHandler,False,"[sfc, pl, hl]"
step,IntHandler,False,"[0, 10000]"
stream,EnumHandler,False,"[wave, oper]"
time,TimeHandler,False,
type,StringHandler,False,


payload required: False


## 3. What is on the data bridge right now

Replay rules on this server, and the helpers that deal with them:

* one replay delivers at most **10 000** notifications and then raises
  `HistoryGapError("replay_limit_reached")`; `replay()` pages through it from the last sequence;
* sequence ids are dense per event type, so `latest_sequence()` finds the newest one with ~25
  one-notification probes, and `replay_tail()` fetches only the last few thousand.

From the tail we derive, per announced field:

| column | meaning |
| --- | --- |
| `base_utc` | forecast base time (`date` + `time`) |
| `published_utc` | when the availability was announced (CloudEvent `time`) |
| `transfer_lag_h` | `published_utc - base_utc`: how long after the run the data was on the bridge |
| `expires_utc` | `base_utc + 15 days`: when the bridge drops it |
| `days_left` | time to `expires_utc` from now (negative = already gone) |

In [3]:
def replay(client, *, filter, start=1, max_pages=None):
    out, cursor, pages = [], start, 0
    while True:
        pages += 1
        try:
            with open_stream(client, EVENT, filter=filter, start=cursor, mode="replay_only") as s:
                out.extend(s)
            return out
        except pyaviso.HistoryGapError as exc:
            if "replay_limit" not in str(exc) or not out or out[-1].sequence == cursor:
                raise
            cursor = out[-1].sequence
            if max_pages and pages >= max_pages:
                return out


def has_after(client, seq, filter={"class": "d1"}):
    with open_stream(client, EVENT, filter=filter, start=seq, mode="replay_only") as s:
        return next(iter(s), None) is not None


def latest_sequence(client):
    lo, hi = 0, 1
    while has_after(client, hi):
        lo, hi = hi, hi * 2
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if has_after(client, mid):
            lo = mid
        else:
            hi = mid
    return lo + 1


def replay_tail(client, *, filter, span=3000):
    last = latest_sequence(client)
    return replay(client, filter=filter, start=max(last - span, 0))


def availability_frame(notifications, now=None):
    now = now or datetime.now(timezone.utc)
    rows = []
    for n in notifications:
        ident = n.identifier
        row = {"sequence": n.sequence, "published_utc": (n.cloudevent or {}).get("time")}
        row.update(ident)
        rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["published_utc"] = pd.to_datetime(df["published_utc"], utc=True, format="ISO8601")
    df["base_utc"] = pd.to_datetime(df["date"].astype(str) + df["time"].astype(str).str.zfill(4),
                                    format="%Y%m%d%H%M", utc=True)
    df["step"] = df["step"].astype(int)
    df["transfer_lag_h"] = (df["published_utc"] - df["base_utc"]).dt.total_seconds() / 3600
    df["expires_utc"] = df["base_utc"] + pd.to_timedelta(RETENTION_DAYS, unit="D")
    df["days_left"] = (df["expires_utc"] - now).dt.total_seconds() / 86400
    return df


TAIL_SPAN = 3000
t0 = time.monotonic()
tail = replay_tail(client, filter={"class": "d1"}, span=TAIL_SPAN)
df = availability_frame(tail)
print(f"{len(tail)} notifications in the last {TAIL_SPAN} sequences "
      f"(seq {tail[0].sequence}..{tail[-1].sequence}) fetched in {time.monotonic() - t0:.1f}s")
print("published between", df["published_utc"].min(), "and", df["published_utc"].max())
df[["sequence", "published_utc", "stream", "levtype", "date", "time", "step", "transfer_lag_h", "expires_utc", "days_left"]].tail(8)

3000 notifications in the last 3000 sequences (seq 136691..139690) fetched in 8.2s
published between 2026-09-16 12:18:10.749033794+00:00 and 2026-09-16 12:18:13.780105777+00:00


,sequence,published_utc,stream,levtype,date,time,step,transfer_lag_h,expires_utc,days_left
2992,139683,2026-09-16 12:18:13.779657342+00:00,wave,sfc,20260915,0000,9,36.303828,2026-09-30 00:00:00+00:00,5.345632
2993,139684,2026-09-16 12:18:13.779670822+00:00,wave,sfc,20260915,0000,93,36.303828,2026-09-30 00:00:00+00:00,5.345632
2994,139685,2026-09-16 12:18:13.779677022+00:00,wave,sfc,20260915,0000,94,36.303828,2026-09-30 00:00:00+00:00,5.345632
2995,139686,2026-09-16 12:18:13.779683354+00:00,wave,sfc,20260915,0000,96,36.303828,2026-09-30 00:00:00+00:00,5.345632
2996,139687,2026-09-16 12:18:13.779854819+00:00,wave,sfc,20260915,0000,92,36.303828,2026-09-30 00:00:00+00:00,5.345632
2997,139688,2026-09-16 12:18:13.779870949+00:00,wave,sfc,20260915,0000,90,36.303828,2026-09-30 00:00:00+00:00,5.345632
2998,139689,2026-09-16 12:18:13.779877600+00:00,wave,sfc,20260915,0000,91,36.303828,2026-09-30 00:00:00+00:00,5.345632
2999,139690,2026-09-16 12:18:13.780105777+00:00,wave,sfc,20260915,0000,95,36.303828,2026-09-30 00:00:00+00:00,5.345632


In [4]:
# "Available now" view: one row per forecast run and product family, with the window it is (was) available.
avail = (df.groupby(["date", "time", "stream", "levtype"])
           .agg(fields=("sequence", "size"), steps=("step", lambda s: f"{s.min()}..{s.max()}"),
                announced_utc=("published_utc", "max"), expires_utc=("expires_utc", "first"),
                days_left=("days_left", "first"))
           .reset_index().sort_values(["date", "time", "stream", "levtype"], ascending=[False, False, True, True]))
avail["status"] = avail["days_left"].apply(lambda d: "on the bridge" if d > 0 else "expired")
display(avail.head(12))
print("still inside the 15-day window:", int((avail["days_left"] > 0).sum()), "of", len(avail), "run/product groups")

,date,time,stream,levtype,fields,steps,announced_utc,expires_utc,days_left,status
28,20260915,0000,oper,hl,97,0..96,2026-09-16 12:18:13.683810319+00:00,2026-09-30 00:00:00+00:00,5.345632,on the bridge
29,20260915,0000,oper,pl,97,0..96,2026-09-16 12:18:13.683789589+00:00,2026-09-30 00:00:00+00:00,5.345632,on the bridge
30,20260915,0000,oper,sfc,97,0..96,2026-09-16 12:18:13.684049367+00:00,2026-09-30 00:00:00+00:00,5.345632,on the bridge
31,20260915,0000,wave,sfc,97,0..96,2026-09-16 12:18:13.780105777+00:00,2026-09-30 00:00:00+00:00,5.345632,on the bridge
24,20260914,0000,oper,hl,97,0..96,2026-09-16 12:18:13.165539082+00:00,2026-09-29 00:00:00+00:00,4.345632,on the bridge
25,20260914,0000,oper,pl,97,0..96,2026-09-16 12:18:13.165249982+00:00,2026-09-29 00:00:00+00:00,4.345632,on the bridge
26,20260914,0000,oper,sfc,97,0..96,2026-09-16 12:18:13.165387087+00:00,2026-09-29 00:00:00+00:00,4.345632,on the bridge
27,20260914,0000,wave,sfc,97,0..96,2026-09-16 12:18:13.217818116+00:00,2026-09-29 00:00:00+00:00,4.345632,on the bridge
20,20260913,0000,oper,hl,97,0..96,2026-09-16 12:18:12.814304328+00:00,2026-09-28 00:00:00+00:00,3.345632,on the bridge
21,20260913,0000,oper,pl,97,0..96,2026-09-16 12:18:12.814314458+00:00,2026-09-28 00:00:00+00:00,3.345632,on the bridge


still inside the 15-day window: 24 of 32 run/product groups


> **What the stream held on 2026-09-24.** The LUMI Aviso 2 `data` stream contained ~140 000
> events, all published within a few hours on **2026-09-16** as a back-fill covering forecast
> dates from 2025-09-01 up to 2026-09-15 (four product families: `oper` sfc/pl/hl and `wave`
> sfc, 97 steps each, 00 UTC). The tail above therefore shows the runs of 8 to 15 September:
> the last of them still inside the 15-day window, the first already expired. Because of the
> back-fill, `transfer_lag_h` is the delay of the *announcement*, not a real transfer time. Once
> the producer publishes as data lands, the same columns give the real lag and remaining time.

## 4. Check against Polytope: inside and outside the window

Aviso tells you the data *was* transferred; the data bridge retention decides whether it is
*still* there. Two single-field requests (2 m temperature, step 0) show both sides: the **newest**
run the stream announces (inside the window) and the **oldest** run in the tail (past it). When
Polytope refuses, its error text states the retention rule itself, for example
`Date is too old, expected > 20260909` observed on 2026-09-24.

In [5]:
import earthkit.data as ekd


def polytope_check(date, time_="0000", step="0", stream="oper", levtype="sfc", param="167"):
    req = {"class": "d1", "dataset": DATASET, "expver": "0001", "stream": stream, "type": "fc",
           "levtype": levtype, "date": str(date), "time": time_, "step": str(step), "param": param}
    t0 = time.monotonic()
    try:
        data = ekd.from_source("polytope", "destination-earth", req, address=POLYTOPE_ADDRESS, stream=False)
        fields = data.to_fieldlist() if hasattr(data, "to_fieldlist") else data
        return {"date": date, "ok": True, "fields": len(fields), "seconds": round(time.monotonic() - t0, 1), "message": ""}
    except Exception as exc:
        text = str(exc)
        detail = text[text.find("Details:"):][:200].replace("\n", " ") if "Details:" in text else text[:200]
        return {"date": date, "ok": False, "fields": 0, "seconds": round(time.monotonic() - t0, 1), "message": detail}


import logging
logging.getLogger("polytope").setLevel(logging.WARNING)      # quieten the polytope-client progress log

by_base = df.sort_values("base_utc")
newest_announced, oldest_in_tail = by_base["date"].iloc[-1], by_base["date"].iloc[0]
checks = pd.DataFrame([polytope_check(newest_announced), polytope_check(oldest_in_tail)])
checks.insert(1, "days_left", [df.loc[df["date"] == d, "days_left"].iloc[0].round(1) for d in checks["date"]])
display(checks)

2026-09-24 17:42:18 - INFO - Key read from /Users/mauf/.polytopeapirc


2026-09-24 17:42:18 - INFO - Sending request...
{'request': 'class: d1\n'
            'dataset: extremes-dt\n'
            "date: '20260915'\n"
            "expver: '0001'\n"
            'levtype: sfc\n'
            "param: '167'\n"
            "step: '0'\n"
            'stream: oper\n'
            "time: '0000'\n"
            'type: fc\n',
 'verb': 'retrieve'}


2026-09-24 17:42:18 - INFO - Polytope user key found in session cache for user mauf


2026-09-24 17:42:19 - INFO - Request accepted. Please poll ./01ejwrk98382znp00bnpagsxc9 for status


2026-09-24 17:42:19 - INFO - Polytope user key found in session cache for user mauf


...

01ejwrk98382znp00bnpagsxc9.grib:   0%|          | 0.00/49.4M [00:00<?, ?B/s]

2026-09-24 17:42:28 - INFO - Key read from /Users/mauf/.polytopeapirc


2026-09-24 17:42:28 - INFO - Sending request...
{'request': 'class: d1\n'
            'dataset: extremes-dt\n'
            "date: '20260908'\n"
            "expver: '0001'\n"
            'levtype: sfc\n'
            "param: '167'\n"
            "step: '0'\n"
            'stream: oper\n'
            "time: '0000'\n"
            'type: fc\n',
 'verb': 'retrieve'}


2026-09-24 17:42:28 - INFO - Polytope user key found in session cache for user mauf


,date,days_left,ok,fields,seconds,message
0,20260915,5.3,True,1,10.8,
1,20260908,-1.7,False,0,0.3,Details: Your request could not be processed: ...


## 5. Integration patterns for business systems

A notification is only useful if it reaches the system that acts on it. The client runs
**triggers** for every matching notification *before* handing it to your code; the same trigger
objects work in replay and live watch, and the CLI accepts the same kinds in YAML.

| trigger | when to use |
| --- | --- |
| `webhook(url, method=, headers=, body_template=)` | call your own HTTP endpoint (order system, scheduler); body templated from the notification |
| `post(url)` | forward the original CloudEvent envelope to a CloudEvents-aware receiver |
| `teams(url)` | Microsoft Teams incoming webhook card |
| `command(cmd)` | run a script; fields available as `{{ notification.identifier.date }}` templates (`{{ notification.identifier }}` renders the whole identifier as compact JSON, `{{ env.NAME }}` an environment variable) |
| `log(path)` | append NDJSON, useful as an audit trail |

Below, the trigger objects are built (no external URL is called: the webhook points at a
placeholder unless `WEBHOOK_URL` is set) and a `command` trigger appends one line per event to a
CSV during a short replay.

In [6]:
WEBHOOK_URL = os.environ.get("WEBHOOK_URL")   # e.g. https://your-company.example/hooks/destine-availability
triggers_available = {
    "webhook": pyaviso.Trigger.webhook(
        WEBHOOK_URL or "https://example.invalid/hooks/destine",
        method="POST", headers={"Content-Type": "application/json"},
        body_template='{"field": {{ notification.identifier }}, "sequence": {{ notification.sequence }}, '
                      '"event_type": "{{ notification.event_type }}"}',
        required=False, timeout=10.0),
    "post":    pyaviso.Trigger.post(WEBHOOK_URL or "https://example.invalid/cloudevents", required=False),
    "teams":   pyaviso.Trigger.teams(os.environ.get("TEAMS_WEBHOOK_URL", "https://example.invalid/teams"), required=False),
    "log":     pyaviso.Trigger.log(OUT_DIR / "events.ndjson"),
}
print({k: type(v).__name__ for k, v in triggers_available.items()})

csv_path = OUT_DIR / "announced.csv"
csv_path.unlink(missing_ok=True)
announce = pyaviso.Trigger.command(
    'echo "{{ notification.sequence }},{{ notification.identifier.date }},{{ notification.identifier.time }},'
    '{{ notification.identifier.stream }},{{ notification.identifier.levtype }},{{ notification.identifier.step }}" >> '
    + str(csv_path.resolve()), timeout=10.0)

# Replay only the very last few sequences through the command + log triggers.
last = latest_sequence(client)
with open_stream(client, EVENT, filter={"class": "d1"}, start=max(last - 12, 0), mode="replay_only",
                 triggers=[announce, triggers_available["log"]]) as s:
    few = list(s)
print(f"{len(few)} notification(s) replayed; {csv_path} has {sum(1 for _ in csv_path.open())} line(s)")
print(csv_path.read_text(encoding="utf-8")[:400])

{'webhook': 'Trigger', 'post': 'Trigger', 'teams': 'Trigger', 'log': 'Trigger'}


12 notification(s) replayed; output/uc03/announced.csv has 12 line(s)
139679,20260915,0000,wave,sfc,81
139680,20260915,0000,wave,sfc,84
139681,20260915,0000,wave,sfc,89
139682,20260915,0000,wave,sfc,88
139683,20260915,0000,wave,sfc,9
139684,20260915,0000,wave,sfc,93
139685,20260915,0000,wave,sfc,94
139686,20260915,0000,wave,sfc,96
139687,20260915,0000,wave,sfc,92
139688,20260915,0000,wave,sfc,90
139689,20260915,0000,wave,sfc,91
139690,20260915,0000,wave,sfc,95



### Resume across restarts

A `JsonFileStore` commits the last delivered sequence per (server, event, filter);
`flush_cursor_on_exit=True` also commits the final position when the stream closes. The file
is created lazily on the first commit, and the cursor only moves forward.

Observed with `pyaviso` 2.1.0 on this server: an explicit `start=` always wins over the stored
cursor, and a Python watch started *without* `start=` began at the live edge rather than at the
cursor, whereas the CLI (`aviso listen --state-file ...`) did resume from it. Until that is
settled, a Python service should read the cursor itself and pass it as `start=`, as the cell
does on its second run; the CLI listener in the next section resumes on its own.

In [7]:
state_path = OUT_DIR / "state.json"
resuming = pyaviso.AvisoClient(base_url=BASE_URL, auth=pyaviso.Bearer(TOKEN),
                               state_store=pyaviso.JsonFileStore(state_path), flush_cursor_on_exit=True)
state_path.unlink(missing_ok=True)
RESUME_FILTER = {"class": "d1"}

# First run: replay the last few sequences; the store commits the last delivered one.
with open_stream(resuming, EVENT, filter=RESUME_FILTER, start=max(last - 5, 0), mode="replay_only") as s:
    print(sum(1 for _ in s), "notification(s) delivered on the first run")
state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {"checkpoints": {}}
print(json.dumps(state, indent=2)[:400])


def committed_cursor(state_path):
    # One checkpoint per (server, event, filter) hash; this store is used for a single filter.
    state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}
    seqs = [c["last_committed_sequence"] for c in state.get("checkpoints", {}).values()]
    return max(seqs) if seqs else None


# Second run ("after a restart"): continue explicitly from the committed cursor.
cursor = committed_cursor(state_path)
with open_stream(resuming, EVENT, filter=RESUME_FILTER, start=cursor, mode="replay_only") as s:
    print(sum(1 for _ in s), f"notification(s) newer than the committed cursor {cursor} (nothing published since)")

5 notification(s) delivered on the first run
{
  "version": 1,
  "key_format_version": 1,
  "checkpoints": {
    "8d95ec885acf8e28009f012b723c698fe0d151ee0a33993e51cd9cb9b617b2c9": {
      "last_committed_sequence": 139690,
      "last_event_id": "data@139690"
    }
  }
}
0 notification(s) newer than the committed cursor 139690 (nothing published since)


### CLI listener for an unattended service

The same watch as a long-running process (systemd unit, container). `--from` catches up from a
publish time or sequence, then stays live; the state file makes restarts resume.

```yaml
# dte-availability.yaml
listeners:
  - name: dte-sfc-availability
    event: data
    identifiers:
      class: d1
      stream: oper
      levtype: sfc
    triggers:
      - type: log
        path: output/uc03/events.ndjson
      - type: webhook
        url: "{{ env.WEBHOOK_URL }}"
        method: POST
        headers:
          Authorization: "Bearer {{ env.WEBHOOK_TOKEN }}"
        body_template: '{"field": {{ notification.identifier }}, "sequence": {{ notification.sequence }}}'
        required: false
      - type: command
        command: python track_availability.py --date {{ notification.identifier.date }} --time {{ notification.identifier.time }} --step {{ notification.identifier.step }}
```

```bash
export AVISO_BASE_URL=https://aviso2.lumi.apps.dte.destination-earth.eu
export AVISO_TOKEN="$(python -c 'import json,pathlib;print(json.load(open(pathlib.Path.home()/".polytopeapirc"))["user_key"])')"
aviso listen dte-availability.yaml --state-file ~/.local/state/aviso/dte.json --from 2026-09-24T00:00:00Z
```

Run once with `--from` to catch up; on later starts leave it out and the CLI resumes from the
state file (verified: a cursor of 139685 in the state file made `aviso listen` replay 139686
onwards before going live).

## 6. A minimal availability tracker

Businesses usually want a **current inventory** rather than a stream: which runs are on the
bridge, until when. The tracker below keeps a JSON file keyed by run and product family, adds
every notification it sees, and prunes entries past their expiry. It is the piece a `command`
trigger or the live loop in section 7 would call.

In [8]:
class AvailabilityTracker:
    def __init__(self, path, retention_days=RETENTION_DAYS):
        self.path, self.retention = pathlib.Path(path), timedelta(days=retention_days)
        self.items = json.loads(self.path.read_text(encoding="utf-8")) if self.path.exists() else {}

    @staticmethod
    def key(ident):
        return "{class}/{stream}/{levtype}/{date}T{time}".format(**{k: ident.get(k, "?") for k in ("class", "stream", "levtype", "date", "time")})

    def add(self, notification, now=None):
        ident = notification.identifier
        now = now or datetime.now(timezone.utc)
        base = datetime.strptime(str(ident["date"]) + str(ident["time"]).zfill(4), "%Y%m%d%H%M").replace(tzinfo=timezone.utc)
        entry = self.items.setdefault(self.key(ident), {
            "base_utc": base.isoformat(), "expires_utc": (base + self.retention).isoformat(),
            "first_announced_utc": now.isoformat(), "steps": []})
        if int(ident["step"]) not in entry["steps"]:
            entry["steps"] = sorted(entry["steps"] + [int(ident["step"])])
        entry["last_announced_utc"] = now.isoformat()
        return entry

    def prune(self, now=None):
        now = now or datetime.now(timezone.utc)
        gone = [k for k, v in self.items.items() if datetime.fromisoformat(v["expires_utc"]) <= now]
        for k in gone:
            del self.items[k]
        return gone

    def save(self):
        self.path.write_text(json.dumps(self.items, indent=1), encoding="utf-8")

    def frame(self):
        return pd.DataFrame([{"run": k, **{kk: vv for kk, vv in v.items() if kk != "steps"}, "n_steps": len(v["steps"])}
                             for k, v in self.items.items()])


tracker = AvailabilityTracker(OUT_DIR / "availability.json")
for n in tail:
    tracker.add(n)
print("tracked runs after ingesting the tail:", len(tracker.items))
expired = tracker.prune()
print("pruned as expired:", len(expired), "| still available:", len(tracker.items))
tracker.save()
tracker.frame().head(10) if tracker.items else print("(nothing inside the window; pruned inventory is empty)")

tracked runs after ingesting the tail: 32
pruned as expired: 8 | still available: 24


,run,base_utc,expires_utc,first_announced_utc,last_announced_utc,n_steps
0,d1/oper/sfc/20260910T0000,2026-09-10T00:00:00+00:00,2026-09-25T00:00:00+00:00,2026-09-24T15:38:38.918453+00:00,2026-09-24T15:42:34.427003+00:00,97
1,d1/oper/hl/20260910T0000,2026-09-10T00:00:00+00:00,2026-09-25T00:00:00+00:00,2026-09-24T15:38:38.918495+00:00,2026-09-24T15:42:34.426813+00:00,97
2,d1/oper/pl/20260910T0000,2026-09-10T00:00:00+00:00,2026-09-25T00:00:00+00:00,2026-09-24T15:38:38.918525+00:00,2026-09-24T15:42:34.426945+00:00,97
3,d1/wave/sfc/20260910T0000,2026-09-10T00:00:00+00:00,2026-09-25T00:00:00+00:00,2026-09-24T15:38:38.921610+00:00,2026-09-24T15:42:34.428863+00:00,97
4,d1/oper/sfc/20260911T0000,2026-09-11T00:00:00+00:00,2026-09-26T00:00:00+00:00,2026-09-24T15:38:38.922454+00:00,2026-09-24T15:42:34.432367+00:00,97
5,d1/oper/pl/20260911T0000,2026-09-11T00:00:00+00:00,2026-09-26T00:00:00+00:00,2026-09-24T15:38:38.922519+00:00,2026-09-24T15:42:34.432392+00:00,97
6,d1/oper/hl/20260911T0000,2026-09-11T00:00:00+00:00,2026-09-26T00:00:00+00:00,2026-09-24T15:38:38.922567+00:00,2026-09-24T15:42:34.432340+00:00,97
7,d1/wave/sfc/20260911T0000,2026-09-11T00:00:00+00:00,2026-09-26T00:00:00+00:00,2026-09-24T15:38:38.925430+00:00,2026-09-24T15:42:34.433754+00:00,97
8,d1/oper/hl/20260912T0000,2026-09-12T00:00:00+00:00,2026-09-27T00:00:00+00:00,2026-09-24T15:38:38.926492+00:00,2026-09-24T15:42:34.437521+00:00,97
9,d1/oper/pl/20260912T0000,2026-09-12T00:00:00+00:00,2026-09-27T00:00:00+00:00,2026-09-24T15:38:38.926593+00:00,2026-09-24T15:42:34.437545+00:00,97


## 7. Live listen, time-bounded

A live watch blocks until the next matching notification. Here a worker thread pumps the stream
into a queue and the main thread applies the deadline, feeds the tracker and stops after
`seconds`, so the cell always returns. Raise `seconds` (or run the CLI listener) for a real
service. `start=` replays first and then stays live; here it is set just before the newest
sequence so one historical event demonstrates the handling.

In [9]:
def listen_bounded(client, *, filter, seconds=20, max_events=None, start=None, handler=print):
    stream = open_stream(client, EVENT, filter=filter, start=start)
    q = queue.Queue()

    def pump():
        try:
            for n in stream:
                q.put(n)
        except Exception as exc:
            q.put(exc)
        finally:
            q.put(None)

    worker = threading.Thread(target=pump, daemon=True)
    worker.start()
    handled, deadline, t0 = [], time.monotonic() + seconds, time.monotonic()
    try:
        while (remaining := deadline - time.monotonic()) > 0:
            try:
                item = q.get(timeout=min(remaining, 1.0))
            except queue.Empty:
                continue
            if item is None:
                break
            if isinstance(item, Exception):
                print(f"stream error: {type(item).__name__}: {item}")
                break
            handler(item)
            handled.append(item)
            if max_events and len(handled) >= max_events:
                break
    finally:
        stream.close()
        worker.join(timeout=5)
    print(f"watch closed after {time.monotonic() - t0:.1f}s, {len(handled)} event(s)")
    return handled


def on_availability(n):
    entry = tracker.add(n)
    print(f"seq={n.sequence} {tracker.key(n.identifier)} step={n.identifier['step']} "
          f"-> on the bridge until {entry['expires_utc']}")


got = listen_bounded(client, filter={"class": "d1"}, seconds=20, max_events=3, start=max(last - 3, 0), handler=on_availability)
tracker.prune(); tracker.save()

seq=139688 d1/wave/sfc/20260915T0000 step=90 -> on the bridge until 2026-09-30T00:00:00+00:00
seq=139689 d1/wave/sfc/20260915T0000 step=91 -> on the bridge until 2026-09-30T00:00:00+00:00
seq=139690 d1/wave/sfc/20260915T0000 step=95 -> on the bridge until 2026-09-30T00:00:00+00:00
watch closed after 0.2s, 3 event(s)


## 8. Publisher side: who raises the event

The consumer pattern above does not depend on who publishes. Two producers are plausible:

* **The transfer job on the data bridge** publishes to Aviso 2 the moment a field is written
  (`POST /api/v1/notification`, or `client.notify(event_type="data", identifier={...})`). This
  is what the current LUMI stream models.
* **ECPDS**, which already emits Aviso 1.x `dissemination` notifications when it delivers
  products, could publish the same event to Aviso 2 (dual publishing during the transition).
  Notebook `01-ecpds-aviso2-notifications.ipynb` covers that mapping.

The ECMWF playground shows an alternative shape for the same use case: its `de-scm-data` stream
carries `experiment`/`date`/`time` identifiers plus a **required payload** with an
`output_path`, i.e. the notification also says *where* the data landed. For the data bridge the
"where" is implicit (Polytope with the identifier keys), so the LUMI `data` stream omits it.

## Related Polytope examples

The DestinE Polytope examples repository (`~/Git/polytope/polytope-examples`, GitHub
`destination-earth-digital-twins/polytope-examples`) contains no Aviso code, but it is the
reference for the **data side** of every Aviso workflow: once a notification names a field, these
notebooks show how to fetch exactly that field (or a feature cut from it) with earthkit-data.

| Need | Example |
| --- | --- |
| Full Extremes-DT field, regional `area` and `grid` options | `extremes-dt/full-field/extremes-dt-earthkit-example-domain.ipynb`, `extremes-dt/full-field-post-processing/extremes-dt-earthkit-serverside-interpolation.ipynb` |
| Feature extraction (timeseries, bounding box, polygon/country, vertical profile, trajectory, wave) | `extremes-dt/feature-extraction/*.ipynb` |
| On-Demand Extremes-DT (DEODE) output: `dataset: on-demand-extremes-dt`, `expver: 0099`, per-run `georef` | `on-demand-extremes-dt/full-field/` and `on-demand-extremes-dt/feature-extraction/` |
| DESP token (`~/.polytopeapirc`) | `desp-authentication.py` |

Operational limits from that README: 50 requests per second and at most 5 concurrent downloads per
user; data are served from LUMI (`polytope.lumi.apps.dte.destination-earth.eu`) and MN5
(`polytope.mn5.apps.dte.destination-earth.eu`). Relative dates such as `"date": "-14"` are accepted.
